# SatcomLLM: Retrieval-Augmented Generation on a Cloud GPU

**Goal of this demo:** ask questions about satellite-communications documents and get answers that are **grounded in those documents**, produced by the domain-tuned model **`esa-sceva/llama3-satcom-8b`**.

## The big picture

```
                 ┌──────────────── THIS MACHINE (local) ──────────────-───┐          ┌──── CLOUD GPU ────┐
 INDEXING        │  data/*.md  ──►  chunks  ──────────────────────────────┼──text──►│ Embedding model   │
 (once)          │                              local Qdrant  ◄───────────┼─vectors─│ Qwen3-Embed-4B    │
                 │                                                        │         │                   │
 QUESTION        │  question ─────────────────────────────────────────────┼──text──►│ Embedding model   │
 (every time)    │  local Qdrant: top-k similar chunks  ◄─────────────────┼─vector──│                   │
                 │  prompt = question + retrieved chunks ─────────────────┼────────►│ LLM (Satcom 8B)   │
                 │  grounded answer + sources  ◄──────────────────────────┼─────────│                   │
                 └────────────────────────────────────────────────────────┘         └───────────────────┘
```

**Documents and vectors stay on this machine. The heavy compute runs on the GPU.**

## The stages of this demo

| Stage | What happens | Where it runs |
|-------|--------------|---------------|
| **0. Setup** | Install packages, point the notebook at the GPU, check both models are up | Local (done before the talk) |
| **1. Talk to the LLM** | Ask the model a question with no extra context | Cloud GPU |
| **2. Why RAG?** | What plain LLMs get wrong, and how retrieval fixes it | Theory |
| **3. Build the knowledge base** | Split the documents into chunks, turn each chunk into a vector, store the vectors | Chunking and storage local, embedding on the GPU |
| **4. Ask questions (RAG)** | Find the most relevant chunks, add them to the prompt, generate the answer | Search local, generation on the GPU |
| **Appendix** | GPU hosting guide, storage management, troubleshooting | Reference only |

## Technology stack

- **LLM:** `esa-sceva/llama3-satcom-8b`, served by vLLM (OpenAI-compatible `/v1/chat/completions`)
- **Embeddings:** `Qwen/Qwen3-Embedding-4B`, served by vLLM (`/v1/embeddings`)
- **Vector database:** Qdrant, running locally (`./qdrant_db/`)
- **Chunking:** LangChain header-aware Markdown splitter, with chunks cached in `./chunks_cache/`

---

# Stage 0: Setup

> **Presenter:** run both cells in this stage **before the audience joins**. They install packages and check that both GPU endpoints respond.

Requirements: a `.env` file in this folder with `GPU_LLM_URL`, `GPU_EMBEDDING_URL`, `LLM_MODEL_NAME`, `EMBEDDING_MODEL_NAME` (copy `env.example`). Hosting details are in the **Appendix**.

In [1]:
import subprocess
import sys
import os
from pathlib import Path
import shutil

# Define venv path
VENV_PATH = Path("./venv")
VENV_PYTHON = VENV_PATH / "bin" / "python"
VENV_PIP = VENV_PATH / "bin" / "pip"

def find_system_python():
    """Find a working system Python 3 to create the venv."""
    python_candidates = [
        "/usr/bin/python3",
        "/usr/local/bin/python3",
        "/opt/homebrew/bin/python3",
        shutil.which("python3"),
    ]

    for python_path in python_candidates:
        if python_path and Path(python_path).exists():
            try:
                result = subprocess.run(
                    [python_path, "--version"],
                    capture_output=True, text=True, timeout=5
                )
                if result.returncode == 0 and "Python 3" in result.stdout:
                    return python_path
            except Exception:
                continue
    return sys.executable

# Step 1: Check if venv exists and is valid
venv_valid = False
if VENV_PATH.exists():
    if VENV_PYTHON.exists():
        try:
            result = subprocess.run(
                [str(VENV_PYTHON), "--version"],
                capture_output=True, text=True, timeout=5
            )
            venv_valid = result.returncode == 0
        except Exception:
            venv_valid = False

    if not venv_valid:
        print("Existing venv is broken, removing it...")
        shutil.rmtree(VENV_PATH, ignore_errors=True)

# Step 2: Create virtual environment if needed
if not VENV_PATH.exists():
    print("Creating virtual environment...")
    system_python = find_system_python()
    print(f"  Using Python: {system_python}")
    subprocess.run([system_python, "-m", "venv", str(VENV_PATH)], check=True)
    print(f"Virtual environment created at {VENV_PATH}")
else:
    print(f"Virtual environment already exists at {VENV_PATH}")

# Step 3: Upgrade pip in venv
print("\nUpgrading pip...")
subprocess.run([str(VENV_PIP), "install", "--upgrade", "pip", "-q"], check=True)

# Step 4: Install required packages (no local model runtime)
print("\nInstalling required packages (this may take a few minutes on first run)...")
packages = [
    "requests",
    "qdrant-client",
    "langchain",
    "langchain-community",
    "langchain-text-splitters",
    "tqdm",
    "python-dotenv",
]

subprocess.run([str(VENV_PIP), "install", "-q"] + packages, check=True)
# Repair native pydantic wheels if a previous openai install left them broken
subprocess.run(
    [str(VENV_PIP), "install", "--force-reinstall", "-q", "pydantic-core", "pydantic"],
    check=True,
)
print("All packages installed successfully!")

# Step 5: Add venv to Python path for this notebook session
venv_site_packages = None
for python_dir in (VENV_PATH / "lib").glob("python*"):
    sp = python_dir / "site-packages"
    if sp.exists():
        venv_site_packages = sp
        break

if venv_site_packages and str(venv_site_packages) not in sys.path:
    sys.path.insert(0, str(venv_site_packages))
    print(f"Added {venv_site_packages} to Python path")

print("\nEnvironment setup complete! You can now run the following cells.")

Virtual environment already exists at venv

Upgrading pip...

Installing required packages (this may take a few minutes on first run)...
All packages installed successfully!
Added venv/lib/python3.12/site-packages to Python path

Environment setup complete! You can now run the following cells.


In [2]:
# Ensure venv is in path
import sys
from pathlib import Path

VENV_PATH = Path("./venv")
for python_dir in (VENV_PATH / "lib").glob("python*"):
    sp = python_dir / "site-packages"
    if sp.exists() and str(sp) not in sys.path:
        sys.path.insert(0, str(sp))

import os
import requests
from dotenv import load_dotenv

# env.example = defaults / template. .env = your real URLs (overrides).
# You must still have a .env; placeholder your-gpu-host values are rejected below.
_nb_dir = Path(".").resolve()
load_dotenv(_nb_dir / "env.example")
_env_path = _nb_dir / ".env"
if _env_path.exists():
    load_dotenv(_env_path, override=True)
    print(f"Loaded environment from {_env_path}")
else:
    print("No .env file found. Copy env.example to .env and set the GPU URLs.")

# Storage paths stay local
CHUNKS_CACHE_DIR = Path("./chunks_cache")
CHUNKS_CACHE_DIR.mkdir(exist_ok=True)

# Cloud GPU endpoints (OpenAI-compatible HTTP APIs)
GPU_LLM_URL = os.getenv("GPU_LLM_URL", "").rstrip("/")
GPU_EMBEDDING_URL = os.getenv("GPU_EMBEDDING_URL", "").rstrip("/")
GPU_API_KEY = os.getenv("GPU_API_KEY", "").strip()

LLM_MODEL_NAME = os.getenv("LLM_MODEL_NAME", "esa-sceva/llama3-satcom-8b")
EMBEDDING_MODEL_NAME = os.getenv("EMBEDDING_MODEL_NAME", "Qwen/Qwen3-Embedding-4B")


def _require_url(name, value):
    if not value or "your-gpu-host" in value:
        raise ValueError(
            f"{name} is not set to a real endpoint. In this folder run:\n"
            "  cp env.example .env\n"
            "Then edit .env. On this GPU pod use http://127.0.0.1:8000/v1 and "
            "http://127.0.0.1:8002/v1. From a laptop use the RunPod proxy URLs."
        )
    return value


GPU_LLM_URL = _require_url("GPU_LLM_URL", GPU_LLM_URL)
GPU_EMBEDDING_URL = _require_url("GPU_EMBEDDING_URL", GPU_EMBEDDING_URL)


def gpu_headers():
    headers = {"Content-Type": "application/json"}
    if GPU_API_KEY:
        headers["Authorization"] = f"Bearer {GPU_API_KEY}"
    return headers


print("Configuration:")
print(f"  LLM endpoint:        {GPU_LLM_URL}")
print(f"  Embedding endpoint:  {GPU_EMBEDDING_URL}")
print(f"  LLM model:           {LLM_MODEL_NAME}")
print(f"  Embedding model:     {EMBEDDING_MODEL_NAME}")
print(f"  API key set:         {'yes' if GPU_API_KEY else 'no'}")
print(f"  Chunks cache:        {CHUNKS_CACHE_DIR.resolve()}")
print("\nModels are expected to already be loaded on the GPU. This notebook only calls the APIs.")


# Health check: both servers must list their model
for label, url in [("LLM", GPU_LLM_URL), ("Embeddings", GPU_EMBEDDING_URL)]:
    try:
        r = requests.get(f"{url}/models", headers=gpu_headers(), timeout=10)
        r.raise_for_status()
        served = [m["id"] for m in r.json().get("data", [])]
        print(f"✓ {label} server is up, serving: {served}")
    except Exception as e:
        print(f"✗ {label} server NOT reachable at {url}: {e}")

Loaded environment from /workspace/live-demo/local-demo/.env
Configuration:
  LLM endpoint:        http://127.0.0.1:8000/v1
  Embedding endpoint:  http://127.0.0.1:8002/v1
  LLM model:           esa-sceva/llama3-satcom-8b
  Embedding model:     Qwen/Qwen3-Embedding-4B
  API key set:         no
  Chunks cache:        /workspace/live-demo/local-demo/chunks_cache

Models are expected to already be loaded on the GPU. This notebook only calls the APIs.
✓ LLM server is up, serving: ['esa-sceva/llama3-satcom-8b']
✓ Embeddings server is up, serving: ['Qwen/Qwen3-Embedding-4B']


---
# Stage 1: Talk to the LLM

The model is hosted on the GPU behind an **OpenAI-compatible API**, so we only send plain text over HTTP. Tokenization, inference and decoding all happen on the server, and no model weights exist on this machine.

The cell below defines a small helper and asks the model a question **without any documents**. Keep this answer in mind: we will compare it with the RAG answer later.

In [3]:
def chat_with_gpu_llm(prompt, max_tokens=256, temperature=0.2, top_p=0.9):
    """
    Generate text using the LLM hosted on the cloud GPU.

    Args:
        prompt: Input text prompt
        max_tokens: Maximum tokens to generate
        temperature: Sampling temperature
        top_p: Top-p sampling parameter

    Returns:
        Generated text
    """
    response = requests.post(
        f"{GPU_LLM_URL}/chat/completions",
        headers=gpu_headers(),
        json={
            "model": LLM_MODEL_NAME,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
            "temperature": temperature,
            "top_p": top_p,
        },
        timeout=120,
    )
    response.raise_for_status()
    payload = response.json()
    return (payload["choices"][0]["message"]["content"] or "").strip()



question = "What is SatcomLLM?"
print(f"Question: {question}\n")
print(f"Answer (no RAG):\n{chat_with_gpu_llm(question, max_tokens=200, temperature=0.7)}")

Question: What is SatcomLLM?

Answer (no RAG):
Satcom-LLM stands for Satellite Communications Large Language Model. It is a type of artificial intelligence (AI) model designed to process and generate human-like text based on its understanding of satellite communications. 

Large language models (LLMs) are trained on vast amounts of text data and can generate coherent and context-specific responses to a wide range of inputs. In the case of Satcom-LLM, it would be trained on a large dataset of information related to satellite communications, including technical specifications, regulatory frameworks, industry trends, and more.

The primary purpose of Satcom-LLM is likely to provide expert-level support and knowledge to users, such as satellite engineers, operators, and other professionals working in the field. It could be used for various tasks, including:

1. Answering technical questions: Satcom-LLM could help users find answers to complex technical questions related to satellite commun

---
# Stage 2: Why RAG?

| Problem with a plain LLM | What it means |
|---|---|
| **Hallucinations** | Confident but false answers |
| **Knowledge cutoff** | Knows nothing after its training data |
| **No sources** | You cannot verify where an answer came from |
| **Static** | Updating its knowledge means retraining |

**Retrieval-Augmented Generation (RAG)** looks up relevant passages first and gives them to the model together with the question:

```
Plain LLM:  Question ─────────────────────────────► LLM ──► Answer (may hallucinate)
RAG:        Question ──► find relevant chunks ──► LLM + context ──► Grounded answer + sources
```

RAG has two phases:

1. **Indexing (once, Stage 3):** Documents → Chunks → Embeddings → Vector DB
2. **Querying (every question, Stage 4):** Question → Embedding → Similarity search → Prompt with context → Answer

To update the model's knowledge, you change the documents. The model is never retrained.

---
# Stage 3: Build the Knowledge Base

## 3.1 Load and chunk the documents (local)

We read the Markdown files in `data/` and split them into **chunks**. The splitter first cuts at Markdown headers so that each section stays together, then splits any section longer than 1000 characters, with a 200-character overlap between pieces. Each chunk keeps its section headers as metadata, and we later show those headers as sources.

Chunks are cached as JSON in `./chunks_cache/`, so later runs load instantly.

In [4]:
import os
from pathlib import Path
import json

# Load all markdown documents from the data folder
data_folder = Path("data")

# Check if data folder exists
if not data_folder.exists():
    print(f"Data folder not found: {data_folder}")
    print("Creating sample data folder...")
    data_folder.mkdir(exist_ok=True)
    print("Please add your markdown files to the 'data' folder")
    documents = {}
else:
    # Find all markdown files
    markdown_files = list(data_folder.glob("*.md")) + list(data_folder.glob("*.markdown"))

    if not markdown_files:
        print(f"No markdown files found in {data_folder}")
        print("Please add markdown files to the 'data' folder")
        documents = {}
    else:
        print(f"Found {len(markdown_files)} markdown file(s) in '{data_folder}':")
        for f in markdown_files:
            print(f"  - {f.name}")

        # Load all documents
        documents = {}
        total_chars = 0

        for file_path in markdown_files:
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
                documents[file_path.name] = content
                total_chars += len(content)
                print(f"\n{file_path.name}: {len(content)} characters")

        print(f"\nTotal characters across all documents: {total_chars}")


import json
from pathlib import Path
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Local cache path (this cell does not need the GPU config cell)
if "CHUNKS_CACHE_DIR" not in globals():
    CHUNKS_CACHE_DIR = Path("./chunks_cache")
CHUNKS_CACHE_DIR.mkdir(exist_ok=True)

def chunk_markdown_document(markdown_text, max_chunk_size=1000, chunk_overlap=200):
    """
    Chunk markdown document using header-based splitting.

    Args:
        markdown_text: Markdown formatted text
        max_chunk_size: Maximum chunk size in characters
        chunk_overlap: Overlap between chunks for context preservation

    Returns:
        List of Document objects with content and metadata
    """
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
        ("####", "Header 4"),
    ]

    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on
    )
    md_header_splits = markdown_splitter.split_text(markdown_text)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )

    all_chunks = []
    for idx, doc in enumerate(md_header_splits):
        if len(doc.page_content) > max_chunk_size:
            sub_chunks = text_splitter.split_text(doc.page_content)
            for sub_idx, sub_chunk in enumerate(sub_chunks):
                chunk_doc = Document(
                    page_content=sub_chunk,
                    metadata={
                        **doc.metadata,
                        "chunk_id": f"{idx}_{sub_idx}",
                        "chunk_size": len(sub_chunk),
                    },
                )
                all_chunks.append(chunk_doc)
        else:
            doc.metadata["chunk_id"] = str(idx)
            doc.metadata["chunk_size"] = len(doc.page_content)
            all_chunks.append(doc)

    return all_chunks

def save_chunks_to_file(chunks, filename):
    """Save chunks to a JSON file for later reuse."""
    chunks_data = []
    for chunk in chunks:
        chunks_data.append({
            "page_content": chunk.page_content,
            "metadata": chunk.metadata,
        })

    filepath = CHUNKS_CACHE_DIR / filename
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(chunks_data, f, indent=2, ensure_ascii=False)

    return filepath

def load_chunks_from_file(filename):
    """Load chunks from a saved JSON file."""
    filepath = CHUNKS_CACHE_DIR / filename
    if not filepath.exists():
        return None

    with open(filepath, "r", encoding="utf-8") as f:
        chunks_data = json.load(f)

    chunks = []
    for chunk_data in chunks_data:
        chunk = Document(
            page_content=chunk_data["page_content"],
            metadata=chunk_data["metadata"],
        )
        chunks.append(chunk)

    return chunks

# Chunk all documents (with local caching)
if "documents" in locals() and documents:
    all_chunks = []

    for filename, markdown_text in documents.items():
        cache_filename = f"{Path(filename).stem}_chunks.json"
        cached_chunks = load_chunks_from_file(cache_filename)

        if cached_chunks:
            print(f"\n✓ Loading cached chunks for {filename} ({len(cached_chunks)} chunks)")
            all_chunks.extend(cached_chunks)
        else:
            print(f"\nProcessing {filename}...")
            doc_chunks = chunk_markdown_document(markdown_text)

            for chunk in doc_chunks:
                chunk.metadata["source_file"] = filename

            saved_path = save_chunks_to_file(doc_chunks, cache_filename)
            print(f"  Generated {len(doc_chunks)} chunks")
            print(f"  ✓ Saved to {saved_path}")

            all_chunks.extend(doc_chunks)

    print(f"\n{'=' * 60}")
    print(f"Total chunks across all documents: {len(all_chunks)}")
    print(f"{'=' * 60}")

    if all_chunks:
        print("\nSample chunks:")
        for i, chunk in enumerate(all_chunks[:3]):
            print(f"\n--- Chunk {i + 1} ---")
            print(f"Source: {chunk.metadata.get('source_file', 'unknown')}")
            print(f"Headers: {chunk.metadata.get('Header 1', '')} > {chunk.metadata.get('Header 2', '')}")
            print(f"Content preview: {chunk.page_content[:150]}...")

    chunks = all_chunks
else:
    print("⚠ No documents loaded. Please add markdown files to the 'data' folder.")
    chunks = []

Found 4 markdown file(s) in 'data':
  - db5563aa-30f9-4b6d-a9de-f8a22f5d30f4.md
  - ea51d98e-f644-4fbb-8558-a661ce56cd9e.md
  - ecf0399d-1463-405b-99a3-6878f4828bd7.md
  - dace33f5-f959-4955-bd68-00229c97599e.md

db5563aa-30f9-4b6d-a9de-f8a22f5d30f4.md: 16223 characters

ea51d98e-f644-4fbb-8558-a661ce56cd9e.md: 21191 characters

ecf0399d-1463-405b-99a3-6878f4828bd7.md: 59954 characters

dace33f5-f959-4955-bd68-00229c97599e.md: 26787 characters

Total characters across all documents: 124155

✓ Loading cached chunks for db5563aa-30f9-4b6d-a9de-f8a22f5d30f4.md (22 chunks)

✓ Loading cached chunks for ea51d98e-f644-4fbb-8558-a661ce56cd9e.md (31 chunks)

✓ Loading cached chunks for ecf0399d-1463-405b-99a3-6878f4828bd7.md (86 chunks)

✓ Loading cached chunks for dace33f5-f959-4955-bd68-00229c97599e.md (43 chunks)

Total chunks across all documents: 182

Sample chunks:

--- Chunk 1 ---
Source: db5563aa-30f9-4b6d-a9de-f8a22f5d30f4.md
Headers: MultiEarth 2023 Deforestation Challenge - Team FORE

## 3.2 Embeddings: text → vectors (GPU)

An **embedding** is a list of numbers that captures the *meaning* of a text. Texts with similar meanings get vectors that point in similar directions. That is how we find relevant chunks without keyword matching.

In [5]:
class CloudGPUEmbeddings:
    """Embeddings via an OpenAI-compatible endpoint on the cloud GPU."""

    def __init__(self, base_url, model):
        self.base_url = base_url.rstrip("/")
        self.model = model

    def embed_documents(self, texts):
        if not texts:
            return []
        response = requests.post(
            f"{self.base_url}/embeddings",
            headers=gpu_headers(),
            json={"model": self.model, "input": list(texts)},
            timeout=120,
        )
        response.raise_for_status()
        data = sorted(response.json()["data"], key=lambda item: item.get("index", 0))
        return [item["embedding"] for item in data]

    def embed_query(self, text):
        return self.embed_documents([text])[0]


print(f"Connecting to embedding model on GPU: {EMBEDDING_MODEL_NAME}")
embedding_model = CloudGPUEmbeddings(GPU_EMBEDDING_URL, EMBEDDING_MODEL_NAME)

sample_text = "The SatcomLLM pipeline generates synthetic QA pairs from documents"
sample_embedding = embedding_model.embed_query(sample_text)

print("✓ Cloud GPU embeddings are reachable")
print(f"✓ Model: {embedding_model.model}") 
print(f"✓ Embedding dimension: {len(sample_embedding)}")
print(f"✓ Sample embedding (first 10 values): {sample_embedding[:10]}")

Connecting to embedding model on GPU: Qwen/Qwen3-Embedding-4B
✓ Cloud GPU embeddings are reachable
✓ Model: Qwen/Qwen3-Embedding-4B
✓ Embedding dimension: 2560
✓ Sample embedding (first 10 values): [-0.00029219110729172826, 0.03062162920832634, -0.019401490688323975, 0.01659645512700081, -0.001205288339406252, 0.058438222855329514, 0.0607757531106472, 0.0016874036518856883, 0.023609042167663574, -0.005201001651585102]


## 3.3 Store the vectors in a local Qdrant database

Every chunk is embedded on the GPU in batches of 32, then stored in **local Qdrant** as a *point*: the vector plus the original text and headers.

```
cached chunks (local) ──► GPU embeddings ──► Qdrant points (local, ./qdrant_db/)
```

If the collection already contains data (as it does here), the upload is skipped.

In [9]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import uuid

print("Initializing local Qdrant vector database...")
# qdrant_client = QdrantClient(":memory:")  # In-memory storage (fast, but not persistent)
qdrant_client = QdrantClient(path="./qdrant_db")

collection_name = "satcom_rag_gpu"
embedding_dim = len(embedding_model.embed_query("test"))

print(f"Creating collection: {collection_name}")
print(f"Vector dimension (from GPU embedding model): {embedding_dim}")

try:
    qdrant_client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
    )
    print(f"Collection '{collection_name}' created successfully!")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Collection '{collection_name}' already exists, using existing collection")
    else:
        raise


from tqdm import tqdm

if chunks:
    collection_info = qdrant_client.get_collection(collection_name=collection_name)
    existing_points = collection_info.points_count

    if existing_points > 0:
        print(f"ℹ Collection '{collection_name}' already contains {existing_points} points.")
        print("  Skipping upload - using existing data.")
        print("  (To re-upload: delete ./qdrant_db/ folder and restart)")
        SKIP_UPLOAD = True
    else:
        SKIP_UPLOAD = False

    if not SKIP_UPLOAD:
        print(f"Embedding {len(chunks)} chunks on the cloud GPU and uploading to local Qdrant...")
        print("This may take a moment...\n")

        points = []
        batch_size = 32
        for i in tqdm(range(0, len(chunks), batch_size), desc="Embedding chunks on GPU"):
            batch_chunks = chunks[i:i + batch_size]
            texts = [chunk.page_content for chunk in batch_chunks]
            embeddings = embedding_model.embed_documents(texts)

            for j, (chunk, embedding) in enumerate(zip(batch_chunks, embeddings)):
                point_id = str(uuid.uuid4())
                points.append(
                    PointStruct(
                        id=point_id,
                        vector=list(embedding),
                        payload={
                            "text": chunk.page_content,
                            "metadata": chunk.metadata,
                            "chunk_id": chunk.metadata.get("chunk_id", f"{i + j}"),
                            "source": chunk.metadata.get("source_file"),
                        },
                    )
                )

        print(f"\nUploading {len(points)} points to local Qdrant...")
        qdrant_client.upsert(
            collection_name=collection_name,
            points=points,
        )
        print(f"✓ Successfully uploaded {len(points)} chunks to local Qdrant!")

    collection_info = qdrant_client.get_collection(collection_name=collection_name)
    print("\nCollection info:")
    print(f"  - Points count: {collection_info.points_count}")
    print(f"  - Vector size: {collection_info.config.params.vectors.size}")
else:
    print("⚠ No chunks to upload. Please load documents first.")

Initializing local Qdrant vector database...
Creating collection: satcom_rag_gpu
Vector dimension (from GPU embedding model): 2560
Collection 'satcom_rag_gpu' already exists, using existing collection
ℹ Collection 'satcom_rag_gpu' already contains 182 points.
  Skipping upload - using existing data.
  (To re-upload: delete ./qdrant_db/ folder and restart)

Collection info:
  - Points count: 182
  - Vector size: 2560


---
# Stage 4: Ask Questions (RAG)

## 4.1 Semantic search

1. The question is embedded **on the GPU**.
2. **Local Qdrant** finds the `top_k` chunks whose vectors are closest by cosine similarity.
3. The chunk text and headers are returned. A higher score means a closer match.

In [10]:
def search_knowledge_base(query, top_k=3):
    """
    Search the local knowledge base. Only the query embedding is computed on the GPU.

    Args:
        query: User's question
        top_k: Number of top results to return

    Returns:
        List of relevant documents with scores
    """
    query_embedding = embedding_model.embed_query(query)

    search_results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=top_k,
    )

    results = []
    for result in search_results.points:
        results.append({
            "text": result.payload["text"],
            "metadata": result.payload["metadata"],
            "score": result.score,
        })

    return results

if chunks:
    test_query = "What is SatcomLLM?"
    print(f"Test Query: {test_query}\n")
    print("=" * 80)

    results = search_knowledge_base(test_query, top_k=3)

    for i, result in enumerate(results, 1):
        print(f"\n--- Result {i} (Score: {result['score']:.4f}) ---")
        print(f"Section: {result['metadata'].get('Header 1', '')} > {result['metadata'].get('Header 2', '')}")
        print(f"Text preview: {result['text'][:300]}...")

    print("\n" + "=" * 80)
    print("✓ Search function working correctly!")
else:
    print("⚠ No chunks available for search. Please load documents first.")

Test Query: What is SatcomLLM?


--- Result 1 (Score: 0.5612) ---
Section: Constructing 4D Radio Map in LEO Satellite Networks with Limited Samples > References
Text preview: * [1] [PERSON]. [PERSON], [PERSON], [PERSON], [PERSON], [PERSON], [PERSON], and [PERSON], \"Leo satellite networks assisted geo-distributed data processing,\" _IEEE Wireless Communications Letters_, 2024.
* [2] [PERSON], [PERSON], [PERSON], [PERSON], [PERSON], [PERSON], [PERSON], and [PERSON], \"Gra...

--- Result 2 (Score: 0.5576) ---
Section: Constructing 4D Radio Map in LEO Satellite Networks with Limited Samples > 
Text preview: LEO satellites, radio map, compressive sensing, tensor decomposition, neural network....

--- Result 3 (Score: 0.5274) ---
Section: Constructing 4D Radio Map in LEO Satellite Networks with Limited Samples > References
Text preview: * [63] [PERSON]. [PERSON], [PERSON], [PERSON], [PERSON], [PERSON], and [PERSON], \"Automated federated pipeline for parameter-efficient fine-tuning of large

## 4.2 The complete RAG pipeline

**Retrieve** the top chunks, **augment** the prompt with them, and **generate** the answer on the GPU. The prompt tells the model to answer only from the provided context and to say so when the context does not contain the answer.

Compare this answer with the Stage 1 answer to the same question.

In [11]:
def ask_rag_question(question, top_k=3, max_tokens=256, temperature=0.7, show_sources=True):
    """
    RAG pipeline: retrieve local chunks, generate the answer on the cloud GPU.

    Args:
        question: User's question
        top_k: Number of documents to retrieve
        max_tokens: Maximum tokens for generation
        temperature: Sampling temperature
        show_sources: Whether to display retrieved sources

    Returns:
        Generated answer
    """
    print(f"\n{'=' * 80}")
    print(f"Question: {question}")
    print(f"{'=' * 80}\n")

    print("Searching local knowledge base...")
    retrieved_docs = search_knowledge_base(question, top_k=top_k)

    if not retrieved_docs:
        return "No relevant documents found in the knowledge base."

    if show_sources:
        print(f"\nRetrieved {len(retrieved_docs)} relevant documents:")
        for i, doc in enumerate(retrieved_docs, 1):
            headers = doc["metadata"].get("Header 1", "")
            if doc["metadata"].get("Header 2"):
                headers += f" > {doc['metadata'].get('Header 2')}"
            print(f"  [{i}] {headers} (score: {doc['score']:.3f})")

    context_parts = []
    for i, doc in enumerate(retrieved_docs, 1):
        headers = []
        if doc["metadata"].get("Header 1"):
            headers.append(doc["metadata"]["Header 1"])
        if doc["metadata"].get("Header 2"):
            headers.append(doc["metadata"]["Header 2"])

        section_info = " > ".join(headers) if headers else "General"
        context_parts.append(f"[Document {i}] {section_info}\n{doc['text']}\n")

    context = "\n".join(context_parts)

    enriched_prompt = f"""You are a helpful assistant specializing in satellite communications.

Use the following context from the documentation to answer the question accurately and concisely.
If the answer cannot be found in the context, say so clearly.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    print("\nGenerating answer with the cloud GPU LLM...")
    answer = chat_with_gpu_llm(
        enriched_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
    )

    print(f"\n{'─' * 80}")
    print("ANSWER:")
    print(f"{'─' * 80}")
    print(answer)
    print(f"\n{'=' * 80}\n")

    return answer

if chunks:
    print("Testing Complete Cloud GPU RAG Pipeline")
    print("=" * 80)

    ask_rag_question(
        "What is SatcomLLM?",
        top_k=3,
        temperature=0.7,
        max_tokens=256,
    )
else:
    print("⚠ No chunks available. Please load documents first.")

Testing Complete Cloud GPU RAG Pipeline

Question: What is SatcomLLM?

Searching local knowledge base...

Retrieved 3 relevant documents:
  [1] Constructing 4D Radio Map in LEO Satellite Networks with Limited Samples > References (score: 0.561)
  [2] Constructing 4D Radio Map in LEO Satellite Networks with Limited Samples (score: 0.558)
  [3] Constructing 4D Radio Map in LEO Satellite Networks with Limited Samples > References (score: 0.527)

Generating answer with the cloud GPU LLM...

────────────────────────────────────────────────────────────────────────────────
ANSWER:
────────────────────────────────────────────────────────────────────────────────
Based on the provided context, I couldn't find any reference to "SatcomLLM".




## 4.3 More questions

Edit the list or add your own questions, including ones from the audience.

In [12]:
if chunks:
    test_questions = [
        "What is a LEO satellite?",
        "What is deep learning?",
        "What is Internet of Things?",
    ]

    for question in test_questions:
        ask_rag_question(question, top_k=3, temperature=0.7, max_tokens=256)
        print("\n" + "=" * 80 + "\n")
else:
    print("⚠ No chunks available. Please load documents first.")


Question: What is a LEO satellite?

Searching local knowledge base...

Retrieved 3 relevant documents:
  [1] Constructing 4D Radio Map in LEO Satellite Networks with Limited Samples (score: 0.655)
  [2] Constructing 4D Radio Map in LEO Satellite Networks with Limited Samples > References (score: 0.613)
  [3] Constructing 4D Radio Map in LEO Satellite Networks with Limited Samples > IV Implementation and Experimental setup (score: 0.609)

Generating answer with the cloud GPU LLM...

────────────────────────────────────────────────────────────────────────────────
ANSWER:
────────────────────────────────────────────────────────────────────────────────
Based on the provided context, a LEO (Low Earth Orbit) satellite is not explicitly defined in the given documents. However, it is implied that a LEO satellite is a type of satellite orbiting the Earth at a relatively low altitude.





Question: What is deep learning?

Searching local knowledge base...

Retrieved 3 relevant documents:
  [1]

In [ ]:
# Audience question
ask_rag_question("How do satellites in low Earth orbit differ from geostationary ones?", top_k=3)

---
# Summary

| Step | Component | Runs on |
|------|-----------|---------|
| Chunking + caching | LangChain splitter, `./chunks_cache/` | Local |
| Embeddings | `Qwen/Qwen3-Embedding-4B` (vLLM) | Cloud GPU |
| Vector search | Qdrant, `./qdrant_db/` | Local |
| Generation | `esa-sceva/llama3-satcom-8b` (vLLM) | Cloud GPU |

**Why this split:** the heavy compute runs on the GPU, your documents never go into a cloud database, and the machine running the notebook needs no GPU and no model downloads.

**Next steps:** add documents to `data/`, tune `top_k` and `temperature`, or swap in a larger hosted model through `.env`.

---

# Appendix (reference only, do not run live)

## A. Configuring `.env`


The notebook reads **`.env`**, not `env.example`. `env.example` is only a template (`python-dotenv` ignores it unless we load it as a fallback).

In this folder:

```bash
cp env.example .env
```

Then edit `.env`. If the notebook runs **on the same GPU pod** as `vllm serve`, use loopback:

```bash
GPU_LLM_URL=http://127.0.0.1:8000/v1
GPU_EMBEDDING_URL=http://127.0.0.1:8002/v1
LLM_MODEL_NAME=esa-sceva/llama3-satcom-8b
EMBEDDING_MODEL_NAME=Qwen/Qwen3-Embedding-4B
```

If the notebook runs on your laptop, use the RunPod proxy URLs instead (`https://<POD_ID>-8000.proxy.runpod.net/v1` and `...-8002...`). After saving `.env`, re-run the configuration cell below. That cell is what defines `GPU_EMBEDDING_URL` and `EMBEDDING_MODEL_NAME` for later cells.

The notebook talks to **OpenAI-compatible** HTTP APIs. That is the usual interface when you host models with vLLM, Text Generation Inference, or Text Embeddings Inference.

**Required settings:**

1. **`GPU_LLM_URL`**: chat completions base URL, including `/v1`  
   Example: `http://203.0.113.10:8000/v1` or `https://<POD_ID>-8000.proxy.runpod.net/v1`
2. **`GPU_EMBEDDING_URL`**: embeddings base URL, including `/v1`  
   On a RunPod pod this is usually a **second** port (`8002`), not `8000` or `8001`. See the hosting section below.
3. **`GPU_API_KEY`**: optional. Leave empty if the server does not check auth
4. **`LLM_MODEL_NAME`** / **`EMBEDDING_MODEL_NAME`**: names as registered on the GPU server

**Assumptions:**
- The LLM is already loaded on the GPU
- The embedding model is already loaded on the GPU
- You do not download or run those models in this notebook

Chunk files and the vector database stay local (`./chunks_cache/`, `./qdrant_db/`).

## B. Hosting both models on a RunPod GPU with vLLM

This notebook expects **OpenAI-compatible** URLs, not RunPod serverless (`https://api.runpod.ai/v2/.../run`). Rent a **Pod**, expose HTTP ports `8000` and `8002`, then start two `vllm serve` processes (LLM first, embeddings second).


#### Install vLLM (on the GPU pod, not in this notebook)

Do **not** install vLLM in the laptop `./venv` used by this notebook. Install it only on the RunPod machine that will run the models.

**Preferred: official image (avoids most install errors).** Create the pod with container image `vllm/vllm-openai:latest` (or a pinned tag). That image already has CUDA, PyTorch, and vLLM. Skip pip entirely and go to the `vllm serve` commands below.

**If you install with pip**, use a **fresh** environment and the **prebuilt wheel**. Do not build from source (that compiles CUDA kernels and is where installs usually fail).

```bash
# On the GPU pod. Python 3.10–3.12. Check the driver first:
nvidia-smi

# Fresh env (do not reuse a notebook/transformers venv)
python3 -m venv ~/vllm-env
source ~/vllm-env/bin/activate
pip install -U pip uv

# Prebuilt wheel + matching PyTorch/CUDA. --torch-backend=auto picks the index
# from the installed driver so you do not mix cu118 / cu121 / cu124 wheels.
uv pip install vllm --torch-backend=auto
```

Confirm the import before serving:

```bash
python -c "import vllm; print(vllm.__version__)"
vllm serve --help
```

**Avoid these (common failure modes):**
- `pip install vllm` into an existing env that already has `torch`, `transformers`, or `openai` (broken `pydantic_core` / mismatched CUDA)
- `pip install git+https://github.com/vllm-project/vllm` or `pip install -e .` without `VLLM_USE_PRECOMPILED=1` (long source build, `nvcc` errors)
- Python 3.9 (this laptop venv is 3.9; vLLM wants 3.10+)
- Installing on macOS/CPU — vLLM needs the NVIDIA GPU pod

If `uv pip install vllm --torch-backend=auto` still fails, pin the CUDA index that matches `nvidia-smi` (for example CUDA 12.4 → `uv pip install vllm --torch-backend=cu124`) or switch to the Docker image.

Activate the env in every pod terminal (`vllm` is not on `PATH` otherwise):

```bash
source ~/vllm-env/bin/activate
```

One process serves one model. Use [esa-sceva/llama3-satcom-8b](https://huggingface.co/esa-sceva/llama3-satcom-8b) for generation and [Qwen/Qwen3-Embedding-4B](https://huggingface.co/Qwen/Qwen3-Embedding-4B) for embeddings. Start the LLM first and wait until `curl http://127.0.0.1:8000/v1/models` succeeds, then start embeddings in a second terminal.

```bash
# Terminal 1 — LLM (chat completions)
VLLM_USE_FLASHINFER_SAMPLER=0 vllm serve esa-sceva/llama3-satcom-8b \
  --host 0.0.0.0 \
  --port 8000 \
  --gpu-memory-utilization 0.70 \
  --served-model-name esa-sceva/llama3-satcom-8b

# Terminal 2 — embeddings (only after the LLM is healthy)
vllm serve Qwen/Qwen3-Embedding-4B \
  --host 0.0.0.0 \
  --port 8002 \
  --runner pooling \
  --gpu-memory-utilization 0.28 \
  --max-model-len 8192 \
  --served-model-name Qwen/Qwen3-Embedding-4B
```

`--host 0.0.0.0` is required so the RunPod proxy can reach the servers. Binding to `127.0.0.1` typically returns **502**.

#### Ports (do not use 8001)

RunPod's nginx already listens on **8001** (and often 8081) inside the container. `vllm serve --port 8001` fails with `OSError: [Errno 98] Address already in use`. Serve embeddings on **8002** and add **8002** as an HTTP port in the pod UI so `https://<POD_ID>-8002.proxy.runpod.net` is reachable.

#### vLLM 0.29 embedding flags

`--task embed` is not a `vllm serve` option anymore. For embedding / rerank / reward models use `--runner pooling`. vLLM may log `Resolved --convert auto to --convert embed`; you can pass `--convert embed` explicitly to silence that.

#### FlashInfer on Blackwell (SM 12.x)

On cards such as **RTX PRO 6000 Blackwell** the chat server can load weights and then die during sampler warmup with:

```text
RuntimeError: FlashInfer requires GPUs with sm75 or higher
```

That message is misleading: the GPU is SM 12.0, which is far above sm75. FlashInfer's JIT looks at the **system** CUDA toolkit (often `/usr/local/cuda` → 12.8), not PyTorch's CUDA 13.x. SM 12.x needs CUDA **≥ 12.9**. The real warning appears earlier as `Failed to get device capability: SM 12.x requires CUDA >= 12.9`, then the empty arch list becomes `sm75 or higher`.

Workaround for the **LLM** process: `VLLM_USE_FLASHINFER_SAMPLER=0` (uses PyTorch sampling; quality is unchanged). The embedding server does not sample, so this flag is unnecessary there.

#### Split GPU memory (two processes, one GPU)

`--gpu-memory-utilization` is a fraction of **total** VRAM, not leftover VRAM. The default `0.92` lets the first process reserve almost the whole GPU. The second then fails with:

```text
ValueError: Free memory on device cuda:0 (7.52/94.97 GiB) on startup is less than desired GPU memory utilization (0.92, 87.37 GiB)
```

You cannot fix that by only lowering the embedding flag: a 4B bf16 model needs more than ~8 GiB just for weights, and the LLM will already have taken the rest. Restart the LLM with a smaller reservation, then start embeddings.

On a ~95 GiB GPU the commands above use about **66 GiB** for the 8B chat model and **27 GiB** for the 4B embedder. Adjust the two fractions so they add to less than ~0.90 and each model still fits (8B chat ≈ 15 GiB weights; 4B embed ≈ 8 GiB weights).

**Also cap embedding context.** Qwen3-Embedding-4B defaults to `max_model_len` **40960**. With `--gpu-memory-utilization 0.18` the weights load (~7.6 GiB) and then KV-cache init fails:

```text
Available KV cache memory: -33.95 GiB
ValueError: No available memory for the cache blocks. Try increasing `gpu_memory_utilization`
```

`--gpu-memory-utilization` is a budget for weights **plus** activations **plus** KV cache. A 40k context blows that budget. For this RAG demo, `--max-model-len 8192` is enough for document chunks, and `0.28` leaves room for KV blocks. Do not raise the embedder above leftover free VRAM (`nvidia-smi`); if the startup check fails, lower the LLM fraction first.

Copy `env.example` to `.env` and set the proxy URLs (Pod ID is on the pod page / Connect tab):

```bash
GPU_LLM_URL=https://<POD_ID>-8000.proxy.runpod.net/v1
GPU_EMBEDDING_URL=https://<POD_ID>-8002.proxy.runpod.net/v1
GPU_API_KEY=
LLM_MODEL_NAME=esa-sceva/llama3-satcom-8b
EMBEDDING_MODEL_NAME=Qwen/Qwen3-Embedding-4B
```

`LLM_MODEL_NAME` and `EMBEDDING_MODEL_NAME` must match `--served-model-name`. If you pass `--api-key mysecret` to `vllm serve`, set the same value in `GPU_API_KEY`.

Check that both servers are ready:

```bash
# On the pod
curl http://127.0.0.1:8000/v1/models
curl http://127.0.0.1:8002/v1/models

# From the laptop (after exposing 8000 and 8002 as HTTP ports)
curl https://<POD_ID>-8000.proxy.runpod.net/v1/models
curl https://<POD_ID>-8002.proxy.runpod.net/v1/models
```

An 8B chat model plus a 4B embedder needs enough VRAM for both (or two pods). Do not point both URLs at the same `vllm serve` unless that process actually implements `/v1/embeddings`.

## C. Storage management

**To Free Up Space**:
1. **Clear chunks cache**: delete `./chunks_cache/` (chunks will be re-processed)
2. **Clear Qdrant DB**: delete `./qdrant_db/` (vectors will be re-embedded on the GPU)

**To Persist Data Between Sessions**:
1. Use persistent Qdrant storage: `QdrantClient(path="./qdrant_db")`
2. Chunks are automatically cached in `./chunks_cache/`
3. Documents in `data/` are your source files

Model weights are **not** cached here. They stay on the GPU host.

The cell below shows current disk usage.

In [ ]:
from pathlib import Path

print(f"Chunks cache: {CHUNKS_CACHE_DIR.resolve()}")
if CHUNKS_CACHE_DIR.exists():
    chunk_files = list(CHUNKS_CACHE_DIR.glob("*_chunks.json"))
    chunks_cache_size = sum(f.stat().st_size for f in CHUNKS_CACHE_DIR.rglob("*") if f.is_file())
    print(f"  Files: {len(chunk_files)}")
    print(f"  Size: {chunks_cache_size / (1024 ** 2):.2f} MB")
else:
    print("  (not created yet)")

qdrant_path = Path("./qdrant_db")
print(f"\nLocal Qdrant: {qdrant_path.resolve()}")
if qdrant_path.exists():
    qdrant_size = sum(f.stat().st_size for f in qdrant_path.rglob("*") if f.is_file())
    print(f"  Size: {qdrant_size / (1024 ** 2):.2f} MB")
else:
    print("  (not created yet)")

## D. Quick troubleshooting

| Symptom | Fix |
|---|---|
| `ValueError: GPU_LLM_URL is not set to a real endpoint` | Create `.env` from `env.example` and re-run the Stage 0 config cell |
| Health check shows ✗ / `502 Bad Gateway` | vLLM is not running or is bound to `127.0.0.1`; start it with `--host 0.0.0.0` (Appendix B) |
| `404` on `/embeddings` | `GPU_EMBEDDING_URL` points to the LLM port; the embedder is on **8002** |
| `Storage folder ./qdrant_db is already accessed by another instance` | Another kernel (e.g. the original notebook) holds the DB. Shut that kernel down, or use `QdrantClient(":memory:")` |
| Vector size mismatch in Qdrant | Embedding model changed; delete `./qdrant_db/` and re-run Stage 3 |
| `NameError` (e.g. `embedding_model`, `chunks`) | Cells were skipped. Run all cells top-to-bottom |
| `ModuleNotFoundError` | Re-run the Stage 0 install cell |